# AlphaTrain Pillar 2k: Heavy Value Head + Adversarial Ranking

Fix the "Value Hallucination" — 1,600 sims made 5/7 seeds WORSE.
The value head is overconfident and wrong on novel positions.

**Key changes from 2j:**
1. **Bigger value head**: 32ch conv + 512 hidden (was 8ch + 256) — 7.5x more capacity
2. **Adversarial ranking**: top-1 vs random move (was top-1 vs top-5) — teaches danger
3. **Dropout 0.3**: prevents overconfident memorization
4. Same γ=0.95, categorical 64-bin, max_score=100, val_weight=0.01, 30% endgame

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (UPDATED with new model/dataset)
2. `expert_v2_pairwise_g095.pt.gz` — γ=0.95 data (already on Drive from 2j)
3. `pillar2j_best.pt` — base model (already on Drive from 2j training)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model (always fresh)
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2j_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    print(f'Copying {f}...')
    shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress data (always fresh)
gz_src = os.path.join(DRIVE, 'expert_v2_pairwise_g095.pt.gz')
pt_dst = '/content/alphatrain/data/expert_v2_pairwise_g095.pt'
print('Decompressing expert_v2_pairwise_g095.pt.gz...')
t0 = time.time()
!gzip -dc {gz_src} > {pt_dst}
print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

# Verify model architecture
from alphatrain.model import AlphaTrainNet
m = AlphaTrainNet(value_channels=32, value_hidden=512, value_dropout=0.3)
n = sum(p.numel() for p in m.parameters())
nv = sum(p.numel() for n, p in m.named_parameters() if 'value' in n)
print(f'\nModel: {n:,} params (value head: {nv:,})')
del m

# Verify data
d = torch.load('/content/alphatrain/data/expert_v2_pairwise_g095.pt', weights_only=True, map_location='cpu')
print(f"\nData: {d['boards'].shape[0]:,} states, gamma={d['gamma']}, max_score={d['max_score']}")
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2k: Heavy Value Head + Adversarial Ranking
# Bigger value head (32ch/512h vs 8ch/256h), dropout 0.3
# Top-1 vs random move ranking (not top-1 vs top-5)
# Same gamma=0.95, categorical 64-bin, 30% endgame oversampling
# Warm start from Pillar 2j (keeps policy backbone, reinits value head)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/expert_v2_pairwise_g095.pt \
    --gpu-data --amp --compile \
    --value-bins 64 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/pillar2j_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2k_best.pt \
    --save-dir /content/checkpoints/pillar2k

In [ ]:
# Evaluate policy score
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2k/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2k/{f}'
    dst = f'{DRIVE}/pillar2k_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')